In [16]:
!pip install -q -U google-genai

In [17]:
import google.genai
print("google-genai:", google.genai.__version__)

google-genai: 2.25.0


In [18]:
import os
from getpass import getpass

os.environ["GEMINI_API_KEY"] = getpass("Enter your Gemini API key: ")

Enter your Gemini API key: ··········


In [19]:
from google import genai

client = genai.Client(
    api_key=os.environ["GEMINI_API_KEY"]
)

print("Client created successfully.")

Client created successfully.


In [21]:
import time

for attempt in range(5):
    try:
        interaction = client.interactions.create(
            model="gemini-3.8-flash",
            input="Say hello in one short sentence."
        )

        print("Success!")
        print(interaction.output_text)
        break

    except Exception as e:
        print(f"Attempt {attempt + 1} failed: {e}")

        if attempt < 4:
            print("Waiting 10 seconds before retrying...")
            time.sleep(10)
        else:
            print("All attempts failed.")

Success!
Hello, it is great to meet you!


In [22]:
import re
import json
import time

In [23]:
def scan_vulnerabilities(code: str, language: str = "unknown") -> dict:
    findings = []

    if not code or not code.strip():
        return {
            "status": "clean",
            "language": language,
            "findings": []
        }

    lines = code.splitlines()

    def add_finding(category, severity, line, description, fix):
        findings.append({
            "category": category,
            "severity": severity,
            "line": line,
            "description": description,
            "suggested_fix": fix
        })

    # 1. Hardcoded secrets
    secret_pattern = re.compile(
        r'(?i)(api[_-]?key|password|passwd|secret|token|access[_-]?key)'
        r'\s*[:=]\s*["\'][^"\']+["\']'
    )

    for line_number, line in enumerate(lines, start=1):
        if secret_pattern.search(line):
            add_finding(
                "Hardcoded Secret",
                "High",
                line_number,
                "A password, API key, token, or secret appears to be hardcoded.",
                "Store secrets in environment variables or a secure secret manager."
            )

    # 2. eval()
    for line_number, line in enumerate(lines, start=1):
        if re.search(r'\beval\s*\(', line):
            add_finding(
                "Code Injection",
                "High",
                line_number,
                "eval() can execute dynamically supplied code.",
                "Avoid eval() and use safe parsing or validated input."
            )

    # 3. exec()
    for line_number, line in enumerate(lines, start=1):
        if re.search(r'\bexec\s*\(', line):
            add_finding(
                "Code Injection",
                "High",
                line_number,
                "exec() can execute dynamically supplied code.",
                "Avoid exec() and use safer alternatives."
            )

    # 4. SQL Injection
    sql_pattern = r'(?i)(SELECT|INSERT|UPDATE|DELETE).*(\+|f["\']|format\s*\()'

    for line_number, line in enumerate(lines, start=1):
        if re.search(sql_pattern, line):
            add_finding(
                "SQL Injection",
                "High",
                line_number,
                "SQL appears to be constructed using string concatenation or formatting.",
                "Use parameterized queries or prepared statements."
            )

    # 5. Command Injection
    for line_number, line in enumerate(lines, start=1):
        if re.search(
            r'subprocess\.\w+\s*\(.*shell\s*=\s*True',
            line
        ):
            add_finding(
                "Command Injection",
                "High",
                line_number,
                "subprocess is using shell=True with a potentially untrusted command.",
                "Avoid shell=True and pass commands as a list of arguments."
            )

    # 6. Pickle
    for line_number, line in enumerate(lines, start=1):
        if re.search(r'pickle\.loads?\s*\(', line):
            add_finding(
                "Insecure Deserialization",
                "High",
                line_number,
                "pickle can execute malicious code when loading untrusted data.",
                "Avoid pickle for untrusted data and use a safe format such as JSON."
            )

    # 7. Unsafe YAML
    for line_number, line in enumerate(lines, start=1):
        if (
            re.search(r'yaml\.load\s*\(', line)
            and "SafeLoader" not in line
        ):
            add_finding(
                "Insecure Deserialization",
                "High",
                line_number,
                "yaml.load() may deserialize unsafe YAML content.",
                "Use yaml.safe_load() or SafeLoader."
            )

    # 8. Weak hashing
    for line_number, line in enumerate(lines, start=1):
        if re.search(r'\b(md5|sha1)\s*\(', line, re.IGNORECASE):
            add_finding(
                "Weak Hashing",
                "Medium",
                line_number,
                "MD5 or SHA-1 is weak for security-sensitive password hashing.",
                "Use Argon2, bcrypt, or scrypt for password hashing."
            )

    # 9. Missing input validation
    for line_number, line in enumerate(lines, start=1):
        if re.search(
            r'(request\.(args|form|json)|input\s*\()',
            line
        ):
            nearby_code = "\n".join(
                lines[
                    max(0, line_number - 3):
                    min(len(lines), line_number + 2)
                ]
            )

            if not re.search(
                r'(validate|sanitize|escape|isinstance|isdigit|len\s*\()',
                nearby_code,
                re.IGNORECASE
            ):
                add_finding(
                    "Missing Input Validation",
                    "Medium",
                    line_number,
                    "User-controlled input appears to be used without obvious validation.",
                    "Validate the type, length, format, and allowed values."
                )

    return {
        "status": "clean" if not findings else "vulnerabilities_found",
        "language": language,
        "findings": findings
    }

In [24]:
test_code = """
import subprocess

password = "admin123"

command = input("Enter command: ")

subprocess.run(command, shell=True)
"""

result = scan_vulnerabilities(test_code, "python")

print(json.dumps(result, indent=2))

{
  "status": "vulnerabilities_found",
  "language": "python",
  "findings": [
    {
      "category": "Hardcoded Secret",
      "severity": "High",
      "line": 4,
      "description": "A password, API key, token, or secret appears to be hardcoded.",
      "suggested_fix": "Store secrets in environment variables or a secure secret manager."
    },
    {
      "category": "Command Injection",
      "severity": "High",
      "line": 8,
      "description": "subprocess is using shell=True with a potentially untrusted command.",
      "suggested_fix": "Avoid shell=True and pass commands as a list of arguments."
    },
    {
      "category": "Missing Input Validation",
      "severity": "Medium",
      "line": 6,
      "description": "User-controlled input appears to be used without obvious validation.",
      "suggested_fix": "Validate the type, length, format, and allowed values."
    }
  ]
}


In [25]:
clean_code = """
def add_numbers(a, b):
    return a + b

result = add_numbers(10, 20)

print(result)
"""

result = scan_vulnerabilities(clean_code, "python")

print(json.dumps(result, indent=2))

{
  "status": "clean",
  "language": "python",
  "findings": []
}


In [26]:
test_code = """
import subprocess
import pickle
import hashlib
import sqlite3
import yaml

password = "admin123"
api_key = "MY_SECRET_KEY"

user_input = input("Enter value: ")

result = eval(user_input)

subprocess.run(user_input, shell=True)

data = pickle.loads(user_input)

hash_value = hashlib.md5(password.encode()).hexdigest()

query = "SELECT * FROM users WHERE name = '" + user_input + "'"

config = yaml.load(user_input)
"""

result = scan_vulnerabilities(test_code, "python")

print(json.dumps(result, indent=2))

{
  "status": "vulnerabilities_found",
  "language": "python",
  "findings": [
    {
      "category": "Hardcoded Secret",
      "severity": "High",
      "line": 8,
      "description": "A password, API key, token, or secret appears to be hardcoded.",
      "suggested_fix": "Store secrets in environment variables or a secure secret manager."
    },
    {
      "category": "Hardcoded Secret",
      "severity": "High",
      "line": 9,
      "description": "A password, API key, token, or secret appears to be hardcoded.",
      "suggested_fix": "Store secrets in environment variables or a secure secret manager."
    },
    {
      "category": "Code Injection",
      "severity": "High",
      "line": 13,
      "description": "eval() can execute dynamically supplied code.",
      "suggested_fix": "Avoid eval() and use safe parsing or validated input."
    },
    {
      "category": "SQL Injection",
      "severity": "High",
      "line": 21,
      "description": "SQL appears to be construc

In [27]:
SYSTEM_PROMPT = """
You are an AI-powered security code reviewer.

Your role is to help developers identify security vulnerabilities
in their source code.

You must:

1. Understand the user's security question.
2. Review the supplied source code.
3. Use the local scan_vulnerabilities tool when security
   vulnerabilities need to be checked.
4. Explain detected vulnerabilities in simple language.
5. Provide a practical suggested fix for every finding.
6. Never claim that code is completely secure.
7. Clearly distinguish between detected issues and general advice.

Return the final response using this structure:

Summary Verdict:
Safe / Needs Attention / High Risk

Findings:
For every vulnerability provide:
- Category
- Severity
- Line
- Problem
- Suggested Fix

Explanation:
Explain the security issues in simple language.

If no obvious vulnerabilities are detected, state:
"No obvious security issues were detected by the scanner."
"""

In [28]:
print(SYSTEM_PROMPT)



You are an AI-powered security code reviewer.

Your role is to help developers identify security vulnerabilities
in their source code.

You must:

1. Understand the user's security question.
2. Review the supplied source code.
3. Use the local scan_vulnerabilities tool when security
   vulnerabilities need to be checked.
4. Explain detected vulnerabilities in simple language.
5. Provide a practical suggested fix for every finding.
6. Never claim that code is completely secure.
7. Clearly distinguish between detected issues and general advice.

Return the final response using this structure:

Summary Verdict:
Safe / Needs Attention / High Risk

Findings:
For every vulnerability provide:
- Category
- Severity
- Line
- Problem
- Suggested Fix

Explanation:
Explain the security issues in simple language.

If no obvious vulnerabilities are detected, state:
"No obvious security issues were detected by the scanner."



In [29]:
scan_tool = {
    "type": "function",
    "name": "scan_vulnerabilities",
    "description": (
        "Scan source code for common security vulnerabilities "
        "such as hardcoded secrets, code injection, SQL injection, "
        "command injection, insecure deserialization, weak hashing, "
        "and missing input validation."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "code": {
                "type": "string",
                "description": "The source code that should be scanned."
            },
            "language": {
                "type": "string",
                "description": "The programming language of the source code."
            }
        },
        "required": ["code"]
    }
}

print(json.dumps(scan_tool, indent=2))

{
  "type": "function",
  "name": "scan_vulnerabilities",
  "description": "Scan source code for common security vulnerabilities such as hardcoded secrets, code injection, SQL injection, command injection, insecure deserialization, weak hashing, and missing input validation.",
  "parameters": {
    "type": "object",
    "properties": {
      "code": {
        "type": "string",
        "description": "The source code that should be scanned."
      },
      "language": {
        "type": "string",
        "description": "The programming language of the source code."
      }
    },
    "required": [
      "code"
    ]
  }
}


In [31]:
def security_review(user_query, code, language="unknown"):

    if not code or not code.strip():
        return "Please provide a code snippet for security review."

    user_prompt = (
        "User Security Question:\n"
        + user_query
        + "\n\nProgramming Language:\n"
        + language
        + "\n\nCode Snippet:\n\n"
        + code
        + "\n\nReview this code for security vulnerabilities. "
        + "Use the scan_vulnerabilities tool when appropriate."
    )

    try:
        interaction = client.interactions.create(
            model="gemini-3.8-flash",
            input=user_prompt,
            system_instruction=SYSTEM_PROMPT,
            tools=[scan_tool]
        )

        return interaction

    except Exception as e:
        return "Error while performing security review: " + str(e)

In [32]:
print("security_review function created successfully.")


security_review function created successfully.


In [34]:
def security_review(user_query, code, language="unknown"):
    if not code or not code.strip():
        return "Please provide a code snippet for security review."

    user_prompt = (
        "User Security Question: " + user_query +
        "\n\nProgramming Language: " + language +
        "\n\nCode Snippet:\n" + code +
        "\n\nReview this code for security vulnerabilities. "
        "Use the scan_vulnerabilities tool when appropriate."
    )

    try:
        interaction = client.interactions.create(
            model="gemini-3.8-flash",
            input=user_prompt,
            system_instruction=SYSTEM_PROMPT,
            tools=[scan_tool]
        )

        return interaction

    except Exception as e:
        return "Error while performing security review: " + str(e)

In [35]:
print("security_review function created successfully.")

security_review function created successfully.


In [36]:
test_query = "Check this Python code for security vulnerabilities."

test_code = """
import subprocess

password = "admin123"

command = input("Enter command: ")

subprocess.run(command, shell=True)
"""

result = security_review(
    test_query,
    test_code,
    "python"
)

print(result)

Error while performing security review: Error code: 429 - {'error': {'message': 'Rate limit exceeded for model gemini-3.8-flash (limit: 20 requests per day on Free Tier). Please retry in 59s or upgrade your tier at https://ai.dev/rate-limit.', 'code': 'too_many_requests'}}


In [37]:
test_query = "Check this Python code for security vulnerabilities."

test_code = """
import subprocess

password = "admin123"

command = input("Enter command: ")

subprocess.run(command, shell=True)
"""

scan_result = scan_vulnerabilities(test_code, "python")

print(json.dumps(scan_result, indent=2))

{
  "status": "vulnerabilities_found",
  "language": "python",
  "findings": [
    {
      "category": "Hardcoded Secret",
      "severity": "High",
      "line": 4,
      "description": "A password, API key, token, or secret appears to be hardcoded.",
      "suggested_fix": "Store secrets in environment variables or a secure secret manager."
    },
    {
      "category": "Command Injection",
      "severity": "High",
      "line": 8,
      "description": "subprocess is using shell=True with a potentially untrusted command.",
      "suggested_fix": "Avoid shell=True and pass commands as a list of arguments."
    },
    {
      "category": "Missing Input Validation",
      "severity": "Medium",
      "line": 6,
      "description": "User-controlled input appears to be used without obvious validation.",
      "suggested_fix": "Validate the type, length, format, and allowed values."
    }
  ]
}


In [38]:
test_code = """
import subprocess
import pickle
import hashlib
import yaml

password = "admin123"
api_key = "MY_SECRET_KEY"

user_input = input("Enter value: ")

result = eval(user_input)

subprocess.run(user_input, shell=True)

data = pickle.loads(user_input)

hash_value = hashlib.md5(password.encode()).hexdigest()

query = "SELECT * FROM users WHERE name = '" + user_input + "'"

config = yaml.load(user_input)
"""

scan_result = scan_vulnerabilities(test_code, "python")

print(json.dumps(scan_result, indent=2))

{
  "status": "vulnerabilities_found",
  "language": "python",
  "findings": [
    {
      "category": "Hardcoded Secret",
      "severity": "High",
      "line": 7,
      "description": "A password, API key, token, or secret appears to be hardcoded.",
      "suggested_fix": "Store secrets in environment variables or a secure secret manager."
    },
    {
      "category": "Hardcoded Secret",
      "severity": "High",
      "line": 8,
      "description": "A password, API key, token, or secret appears to be hardcoded.",
      "suggested_fix": "Store secrets in environment variables or a secure secret manager."
    },
    {
      "category": "Code Injection",
      "severity": "High",
      "line": 12,
      "description": "eval() can execute dynamically supplied code.",
      "suggested_fix": "Avoid eval() and use safe parsing or validated input."
    },
    {
      "category": "SQL Injection",
      "severity": "High",
      "line": 20,
      "description": "SQL appears to be construc

In [39]:
clean_code = """
def add(a, b):
    return a + b

result = add(10, 20)

print(result)
"""

scan_result = scan_vulnerabilities(clean_code, "python")

print(json.dumps(scan_result, indent=2))

{
  "status": "clean",
  "language": "python",
  "findings": []
}


In [40]:
def create_security_report(scan_result):
    findings = scan_result.get("findings", [])

    if not findings:
        verdict = "Safe"
    elif any(f["severity"] == "High" for f in findings):
        verdict = "High Risk"
    else:
        verdict = "Needs Attention"

    report = []

    report.append("=" * 60)
    report.append("AI SECURITY CODE REVIEW")
    report.append("=" * 60)
    report.append("")

    report.append("Summary Verdict:")
    report.append(verdict)
    report.append("")

    report.append("Findings:")

    if not findings:
        report.append("No obvious security issues were detected by the scanner.")
    else:
        for i, finding in enumerate(findings, start=1):
            report.append("")
            report.append(f"Finding {i}")
            report.append(f"Category: {finding['category']}")
            report.append(f"Severity: {finding['severity']}")
            report.append(f"Line: {finding['line']}")
            report.append(f"Problem: {finding['description']}")
            report.append(f"Suggested Fix: {finding['suggested_fix']}")

    report.append("")
    report.append("Explanation:")

    if not findings:
        report.append(
            "The scanner did not detect any of the configured "
            "security vulnerability patterns."
        )
    else:
        report.append(
            "The scanner detected one or more security patterns "
            "that may expose the application to attacks. "
            "Review each finding and apply the suggested fixes."
        )

    return "\n".join(report)

In [41]:
test_code = """
import subprocess

password = "admin123"

command = input("Enter command: ")

subprocess.run(command, shell=True)
"""

scan_result = scan_vulnerabilities(test_code, "python")

print(create_security_report(scan_result))

AI SECURITY CODE REVIEW

Summary Verdict:
High Risk

Findings:

Finding 1
Category: Hardcoded Secret
Severity: High
Line: 4
Problem: A password, API key, token, or secret appears to be hardcoded.
Suggested Fix: Store secrets in environment variables or a secure secret manager.

Finding 2
Category: Command Injection
Severity: High
Line: 8
Problem: subprocess is using shell=True with a potentially untrusted command.
Suggested Fix: Avoid shell=True and pass commands as a list of arguments.

Finding 3
Category: Missing Input Validation
Severity: Medium
Line: 6
Problem: User-controlled input appears to be used without obvious validation.
Suggested Fix: Validate the type, length, format, and allowed values.

Explanation:
The scanner detected one or more security patterns that may expose the application to attacks. Review each finding and apply the suggested fixes.


In [42]:
def local_security_review(user_query, code, language="unknown"):
    if not code or not code.strip():
        return "Please provide code for security review."

    scan_result = scan_vulnerabilities(code, language)

    report = create_security_report(scan_result)

    return report

In [43]:
code = """
import subprocess

password = "admin123"

command = input("Enter command: ")

subprocess.run(command, shell=True)
"""

print(
    local_security_review(
        "Find security vulnerabilities in this code.",
        code,
        "python"
    )
)

AI SECURITY CODE REVIEW

Summary Verdict:
High Risk

Findings:

Finding 1
Category: Hardcoded Secret
Severity: High
Line: 4
Problem: A password, API key, token, or secret appears to be hardcoded.
Suggested Fix: Store secrets in environment variables or a secure secret manager.

Finding 2
Category: Command Injection
Severity: High
Line: 8
Problem: subprocess is using shell=True with a potentially untrusted command.
Suggested Fix: Avoid shell=True and pass commands as a list of arguments.

Finding 3
Category: Missing Input Validation
Severity: Medium
Line: 6
Problem: User-controlled input appears to be used without obvious validation.
Suggested Fix: Validate the type, length, format, and allowed values.

Explanation:
The scanner detected one or more security patterns that may expose the application to attacks. Review each finding and apply the suggested fixes.


In [44]:
def run_security_reviewer():
    print("=" * 60)
    print("       AI SECURITY CODE REVIEWER")
    print("=" * 60)

    user_query = input("\nEnter your security question: ")
    language = input("Enter programming language: ")

    print("\nPaste your code below.")
    print("Type END on a new line when finished.\n")

    code_lines = []

    while True:
        line = input()

        if line.strip() == "END":
            break

        code_lines.append(line)

    code = "\n".join(code_lines)

    if not code.strip():
        print("\nNo code was provided.")
        return

    print("\n" + "=" * 60)
    print("SECURITY ANALYSIS")
    print("=" * 60)

    result = local_security_review(
        user_query,
        code,
        language
    )

    print(result)

In [45]:
run_security_reviewer()

       AI SECURITY CODE REVIEWER

Enter your security question: Find security vulnerabilities in this code.
Enter programming language: python

Paste your code below.
Type END on a new line when finished.

import subprocess  password = "admin123"  command = input("Enter command: ")  subprocess.run(command, shell=True)
END

SECURITY ANALYSIS
AI SECURITY CODE REVIEW

Summary Verdict:
High Risk

Findings:

Finding 1
Category: Hardcoded Secret
Severity: High
Line: 1
Problem: A password, API key, token, or secret appears to be hardcoded.
Suggested Fix: Store secrets in environment variables or a secure secret manager.

Finding 2
Category: Command Injection
Severity: High
Line: 1
Problem: subprocess is using shell=True with a potentially untrusted command.
Suggested Fix: Avoid shell=True and pass commands as a list of arguments.

Finding 3
Category: Missing Input Validation
Severity: Medium
Line: 1
Problem: User-controlled input appears to be used without obvious validation.
Suggested Fix: V

In [46]:
run_security_reviewer()

       AI SECURITY CODE REVIEWER

Enter your security question: Check whether this code has security problems.
Enter programming language: python

Paste your code below.
Type END on a new line when finished.

def calculate_sum(a, b):     return a + b  result = calculate_sum(10, 20)  print(result)
END

SECURITY ANALYSIS
AI SECURITY CODE REVIEW

Summary Verdict:
Safe

Findings:
No obvious security issues were detected by the scanner.

Explanation:
The scanner did not detect any of the configured security vulnerability patterns.


In [47]:
def test_eval_detection():
    code = "result = eval(user_input)"

    result = scan_vulnerabilities(code, "python")

    assert any(
        f["category"] == "Code Injection"
        for f in result["findings"]
    )


def test_shell_injection():
    code = "subprocess.run(command, shell=True)"

    result = scan_vulnerabilities(code, "python")

    assert any(
        f["category"] == "Command Injection"
        for f in result["findings"]
    )


def test_sql_injection():
    code = 'query = "SELECT * FROM users WHERE id=" + user_id'

    result = scan_vulnerabilities(code, "python")

    assert any(
        f["category"] == "SQL Injection"
        for f in result["findings"]
    )


def test_clean_code():
    code = """
def add(a, b):
    return a + b
"""

    result = scan_vulnerabilities(code, "python")

    assert result["status"] == "clean"
    assert len(result["findings"]) == 0


print("Running tests...")

test_eval_detection()
test_shell_injection()
test_sql_injection()
test_clean_code()

print("All tests passed!")

Running tests...
All tests passed!


In [48]:
def ai_security_review(user_query, code, language="unknown"):

    if not code or not code.strip():
        return "Please provide code for security review."

    user_prompt = (
        "Security Question: " + user_query +
        "\n\nProgramming Language: " + language +
        "\n\nSource Code:\n" + code +
        "\n\nReview this code for security vulnerabilities. "
        "Use the scan_vulnerabilities tool when appropriate."
    )

    try:
        # First Gemini call
        interaction = client.interactions.create(
            model="gemini-3.8-flash",
            input=user_prompt,
            system_instruction=SYSTEM_PROMPT,
            tools=[scan_tool]
        )

        # Check whether Gemini requested a function
        function_calls = [
            step for step in interaction.steps
            if step.type == "function_call"
        ]

        # Gemini did not request the scanner
        if not function_calls:
            return interaction.output_text

        # Execute requested tools
        function_results = []

        for call in function_calls:

            if call.name == "scan_vulnerabilities":

                arguments = call.arguments

                scanner_result = scan_vulnerabilities(
                    arguments.get("code", code),
                    arguments.get("language", language)
                )

                function_results.append({
                    "type": "function_result",
                    "name": call.name,
                    "call_id": call.id,
                    "result": [
                        {
                            "type": "text",
                            "text": json.dumps(scanner_result)
                        }
                    ]
                })

        # Send scanner result back to Gemini
        final_interaction = client.interactions.create(
            model="gemini-3.8-flash",
            previous_interaction_id=interaction.id,
            input=function_results,
            system_instruction=SYSTEM_PROMPT,
            tools=[scan_tool]
        )

        return final_interaction.output_text

    except Exception as e:
        return "Error while performing AI security review: " + str(e)

In [49]:
print("AI security reviewer created successfully.")

AI security reviewer created successfully.


In [50]:
test_code = """
import subprocess

password = "admin123"

command = input("Enter command: ")

subprocess.run(command, shell=True)
"""

result = ai_security_review(
    "Find security vulnerabilities in this Python code and explain how to fix them.",
    test_code,
    "python"
)

print(result)

Error while performing AI security review: Error code: 429 - {'error': {'message': 'Rate limit exceeded for model gemini-3.8-flash (limit: 20 requests per day on Free Tier). Please retry later or upgrade your tier at https://ai.dev/rate-limit.', 'code': 'too_many_requests'}}


In [51]:
def ai_security_review(user_query, code, language="unknown"):

    if not code or not code.strip():
        return "Please provide code for security review."

    user_prompt = (
        "Security Question: " + user_query +
        "\n\nProgramming Language: " + language +
        "\n\nSource Code:\n" + code +
        "\n\nReview this code for security vulnerabilities. "
        "Use the scan_vulnerabilities tool when appropriate."
    )

    try:
        interaction = client.interactions.create(
            model="gemini-3.8-flash",
            input=user_prompt,
            system_instruction=SYSTEM_PROMPT,
            tools=[scan_tool]
        )

        function_calls = [
            step for step in interaction.steps
            if step.type == "function_call"
        ]

        if not function_calls:
            return interaction.output_text

        function_results = []

        for call in function_calls:

            if call.name == "scan_vulnerabilities":

                arguments = call.arguments

                scanner_result = scan_vulnerabilities(
                    arguments.get("code", code),
                    arguments.get("language", language)
                )

                function_results.append({
                    "type": "function_result",
                    "name": call.name,
                    "call_id": call.id,
                    "result": [
                        {
                            "type": "text",
                            "text": json.dumps(scanner_result)
                        }
                    ]
                })

        final_interaction = client.interactions.create(
            model="gemini-3.8-flash",
            previous_interaction_id=interaction.id,
            input=function_results,
            system_instruction=SYSTEM_PROMPT,
            tools=[scan_tool]
        )

        return final_interaction.output_text

    except Exception as e:

        error_message = str(e)

        if "429" in error_message or "too_many_requests" in error_message:
            return (
                "AI REVIEW TEMPORARILY UNAVAILABLE\n\n"
                "Reason: Gemini API Free Tier request limit has been reached.\n\n"
                "The local security scanner is still available. "
                "Please try the AI review again after the API quota resets."
            )

        elif "503" in error_message or "UNAVAILABLE" in error_message:
            return (
                "AI REVIEW TEMPORARILY UNAVAILABLE\n\n"
                "Gemini is currently experiencing high demand. "
                "Please try again later."
            )

        else:
            return (
                "Error while performing AI security review:\n"
                + error_message
            )

In [52]:
test_code = """
import subprocess

password = "admin123"

command = input("Enter command: ")

subprocess.run(command, shell=True)
"""

scan_result = scan_vulnerabilities(test_code, "python")

print(create_security_report(scan_result))

AI SECURITY CODE REVIEW

Summary Verdict:
High Risk

Findings:

Finding 1
Category: Hardcoded Secret
Severity: High
Line: 4
Problem: A password, API key, token, or secret appears to be hardcoded.
Suggested Fix: Store secrets in environment variables or a secure secret manager.

Finding 2
Category: Command Injection
Severity: High
Line: 8
Problem: subprocess is using shell=True with a potentially untrusted command.
Suggested Fix: Avoid shell=True and pass commands as a list of arguments.

Finding 3
Category: Missing Input Validation
Severity: Medium
Line: 6
Problem: User-controlled input appears to be used without obvious validation.
Suggested Fix: Validate the type, length, format, and allowed values.

Explanation:
The scanner detected one or more security patterns that may expose the application to attacks. Review each finding and apply the suggested fixes.


In [53]:
README = """
# AI-Powered Security Code Reviewer

## 1. Project Overview

This project is an AI-powered security code review application.
It accepts a user's security question and source code, analyzes
the code for common security vulnerabilities, and provides
security findings with severity levels and suggested fixes.

The application uses Gemini as the AI security reviewer and a
local Python function called `scan_vulnerabilities` as the
security scanning tool.

---

## 2. Objectives

- Analyze source code for common security vulnerabilities.
- Use an AI model to understand the developer's security question.
- Automatically use the local vulnerability scanner when required.
- Explain security issues in simple language.
- Provide practical fixes for detected vulnerabilities.
- Handle clean code and API errors gracefully.

---

## 3. Technologies Used

- Python
- Google Gemini API
- Google GenAI Python SDK
- Google Colab
- Regular Expressions
- JSON

---

## 4. Security Vulnerabilities Detected

The local scanner checks for:

1. Hardcoded secrets
2. Code injection through eval()
3. Code injection through exec()
4. SQL injection
5. Command injection
6. Insecure deserialization using pickle
7. Unsafe YAML loading
8. Weak hashing using MD5 or SHA-1
9. Missing input validation

---

## 5. System Architecture

User
 |
 v
Security Question + Source Code
 |
 v
Gemini AI Security Reviewer
 |
 v
Tool Decision
 |
 v
scan_vulnerabilities()
 |
 v
Structured Vulnerability Findings
 |
 v
Gemini
 |
 v
Final Security Report

---

## 6. Main Components

### Gemini AI Reviewer

Gemini understands the user's security question and reviews the
submitted source code.

### scan_vulnerabilities()

This is a local Python security scanning function. It analyzes
source code using regular-expression-based security heuristics.

### Security Report

The final report contains:

- Summary verdict
- Vulnerability category
- Severity
- Line number
- Problem description
- Suggested fix
- Plain-language explanation

---

## 7. Risk Levels

### Safe

No configured vulnerability patterns were detected.

### Needs Attention

Medium-severity security issues were detected.

### High Risk

One or more high-severity security issues were detected.

---

## 8. Example Vulnerable Code

```python
import subprocess

password = "admin123"

command = input("Enter command: ")

subprocess.run(command, shell=True)

_IncompleteInputError: incomplete input (2590815890.py, line 1)

In [54]:
from google.colab import files

README = [
    "# AI-Powered Security Code Reviewer",
    "",
    "## 1. Project Overview",
    "An AI-powered Python application that reviews source code for common security vulnerabilities.",
    "The application combines a local rule-based security scanner with an AI model.",
    "",
    "## 2. Objectives",
    "- Detect common security vulnerabilities.",
    "- Explain vulnerabilities in simple language.",
    "- Provide suggested fixes.",
    "- Use an AI model to decide when the local scanner should be called.",
    "- Handle API and tool errors gracefully.",
    "",
    "## 3. Technologies Used",
    "- Python 3",
    "- Google Gemini API",
    "- Google GenAI SDK",
    "- Regular Expressions (Regex)",
    "- JSON",
    "- Google Colab",
    "",
    "## 4. Vulnerabilities Detected",
    "The local scanner checks for:",
    "",
    "1. Hardcoded secrets and credentials",
    "2. eval() and exec() code injection",
    "3. SQL injection",
    "4. Command injection using shell=True",
    "5. Insecure deserialization using pickle",
    "6. Unsafe YAML loading",
    "7. Weak MD5 and SHA-1 hashing",
    "8. Missing input validation",
    "",
    "## 5. System Architecture",
    "",
    "User Query + Source Code",
    "        |",
    "        v",
    "AI Security Reviewer",
    "        |",
    "        v",
    "scan_vulnerabilities Tool",
    "        |",
    "        v",
    "Security Findings",
    "        |",
    "        v",
    "Final Security Report",
    "",
    "## 6. Risk Levels",
    "",
    "- High: Vulnerabilities that may directly allow attacks.",
    "- Medium: Security weaknesses that should be reviewed and fixed.",
    "- Safe: No configured vulnerability patterns were detected.",
    "",
    "## 7. Example Vulnerable Code",
    "",
    "```python",
    "user_input = input('Enter expression: ')",
    "result = eval(user_input)",
    "```",
    "",
    "The scanner identifies eval() as a potential code injection vulnerability.",
    "",
    "## 8. Example Safe Code",
    "",
    "```python",
    "a = 10",
    "b = 20",
    "result = a + b",
    "```",
    "",
    "No configured security vulnerability is detected in this example.",
    "",
    "## 9. Testing",
    "Automated tests were created for:",
    "- eval() detection",
    "- shell=True detection",
    "- SQL injection detection",
    "- Clean code detection",
    "",
    "## 10. Error Handling",
    "The application handles:",
    "- Empty source code",
    "- Gemini API errors",
    "- Rate-limit errors",
    "- Temporary service availability errors",
    "",
    "## 11. Limitations",
    "- The scanner uses pattern-based detection.",
    "- It may produce false positives or false negatives.",
    "- It does not replace a professional security audit.",
    "- AI availability depends on API limits and service availability.",
    "",
    "## 12. Future Scope",
    "- Add Bandit and other static analysis tools.",
    "- Support more programming languages.",
    "- Add a web-based user interface.",
    "- Generate downloadable security reports.",
    "- Add vulnerability severity scoring.",
    "",
    "## 13. Conclusion",
    "The project demonstrates how an AI model can work with a local security scanning tool",
    "to identify common vulnerabilities and provide understandable security recommendations."
]

with open("README.md", "w", encoding="utf-8") as f:
    f.write("\n".join(README))

print("README.md created successfully!")

files.download("README.md")

README.md created successfully!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [55]:
import os

folders = [
    "code_sentry",
    "code_sentry/tests"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Project folders created successfully!")

Project folders created successfully!


In [56]:
tools_code = r'''
import re


def scan_vulnerabilities(code: str, language: str = "unknown") -> dict:
    findings = []

    if not code or not code.strip():
        return {
            "status": "clean",
            "language": language,
            "findings": []
        }

    lines = code.splitlines()

    def add_finding(category, severity, line, description, fix):
        findings.append({
            "category": category,
            "severity": severity,
            "line": line,
            "description": description,
            "suggested_fix": fix
        })

    # Hardcoded secrets
    secret_pattern = re.compile(
        r'(?i)(api[_-]?key|password|passwd|secret|token|access[_-]?key)'
        r'\s*[:=]\s*["\'][^"\']+["\']'
    )

    for line_number, line in enumerate(lines, start=1):
        if secret_pattern.search(line):
            add_finding(
                "Hardcoded Secret",
                "High",
                line_number,
                "A password, API key, token, or secret appears to be hardcoded.",
                "Store secrets in environment variables or a secure secret manager."
            )

    # eval()
    for line_number, line in enumerate(lines, start=1):
        if re.search(r'\beval\s*\(', line):
            add_finding(
                "Code Injection",
                "High",
                line_number,
                "eval() can execute dynamically supplied code.",
                "Avoid eval() and use safe parsing or validated input."
            )

    # exec()
    for line_number, line in enumerate(lines, start=1):
        if re.search(r'\bexec\s*\(', line):
            add_finding(
                "Code Injection",
                "High",
                line_number,
                "exec() can execute dynamically supplied code.",
                "Avoid exec() and use safer alternatives."
            )

    # SQL Injection
    sql_pattern = r'(?i)(SELECT|INSERT|UPDATE|DELETE).*(\+|f["\']|format\s*\()'

    for line_number, line in enumerate(lines, start=1):
        if re.search(sql_pattern, line):
            add_finding(
                "SQL Injection",
                "High",
                line_number,
                "SQL appears to be constructed using string concatenation or formatting.",
                "Use parameterized queries or prepared statements."
            )

    # Command Injection
    for line_number, line in enumerate(lines, start=1):
        if re.search(
            r'subprocess\.\w+\s*\(.*shell\s*=\s*True',
            line
        ):
            add_finding(
                "Command Injection",
                "High",
                line_number,
                "subprocess is using shell=True with a potentially untrusted command.",
                "Avoid shell=True and pass commands as a list of arguments."
            )

    # Pickle
    for line_number, line in enumerate(lines, start=1):
        if re.search(r'pickle\.loads?\s*\(', line):
            add_finding(
                "Insecure Deserialization",
                "High",
                line_number,
                "pickle can execute malicious code when loading untrusted data.",
                "Avoid pickle for untrusted data and use a safe format such as JSON."
            )

    # Unsafe YAML
    for line_number, line in enumerate(lines, start=1):
        if re.search(r'yaml\.load\s*\(', line) and "SafeLoader" not in line:
            add_finding(
                "Insecure Deserialization",
                "High",
                line_number,
                "yaml.load() may deserialize unsafe YAML content.",
                "Use yaml.safe_load() or SafeLoader."
            )

    # Weak hashing
    for line_number, line in enumerate(lines, start=1):
        if re.search(r'\b(md5|sha1)\s*\(', line, re.IGNORECASE):
            add_finding(
                "Weak Hashing",
                "Medium",
                line_number,
                "MD5 or SHA-1 is weak for security-sensitive password hashing.",
                "Use Argon2, bcrypt, or scrypt for password hashing."
            )

    # Missing input validation
    for line_number, line in enumerate(lines, start=1):
        if re.search(r'(request\.(args|form|json)|input\s*\()', line):
            nearby_code = "\n".join(
                lines[max(0, line_number - 3):min(len(lines), line_number + 2)]
            )

            if not re.search(
                r'(validate|sanitize|escape|isinstance|isdigit|len\s*\()',
                nearby_code,
                re.IGNORECASE
            ):
                add_finding(
                    "Missing Input Validation",
                    "Medium",
                    line_number,
                    "User-controlled input appears to be used without obvious validation.",
                    "Validate the type, length, format, and allowed values."
                )

    return {
        "status": "clean" if not findings else "vulnerabilities_found",
        "language": language,
        "findings": findings
    }
'''

with open("code_sentry/tools.py", "w", encoding="utf-8") as f:
    f.write(tools_code)

print("tools.py created successfully!")

tools.py created successfully!


In [57]:
import sys

if "/content" not in sys.path:
    sys.path.append("/content")

from code_sentry.tools import scan_vulnerabilities

test_code = """
user_input = input("Enter expression: ")
result = eval(user_input)
"""

result = scan_vulnerabilities(test_code, "python")

print(result)

{'status': 'vulnerabilities_found', 'language': 'python', 'findings': [{'category': 'Code Injection', 'severity': 'High', 'line': 3, 'description': 'eval() can execute dynamically supplied code.', 'suggested_fix': 'Avoid eval() and use safe parsing or validated input.'}, {'category': 'Missing Input Validation', 'severity': 'Medium', 'line': 2, 'description': 'User-controlled input appears to be used without obvious validation.', 'suggested_fix': 'Validate the type, length, format, and allowed values.'}]}


In [58]:
PROMPTS = r'''
SYSTEM_PROMPT = """
You are an AI-powered security code reviewer.

Your role is to help developers identify security vulnerabilities
in their source code.

You must:

1. Understand the user's security question.
2. Review the supplied source code.
3. Use the local scan_vulnerabilities tool when security
   vulnerabilities need to be checked.
4. Explain detected vulnerabilities in simple language.
5. Provide a practical suggested fix for every finding.
6. Never claim that code is completely secure.
7. Clearly distinguish detected issues from general advice.

Return the final response using this structure:

Summary Verdict:
Safe / Needs Attention / High Risk

Findings:
For every vulnerability provide:
- Category
- Severity
- Line
- Problem
- Suggested Fix

Explanation:
Explain the security issues in simple language.

If no obvious vulnerabilities are detected, state:
"No obvious security issues were detected by the scanner."
"""
'''

with open("code_sentry/prompts.py", "w", encoding="utf-8") as f:
    f.write(PROMPTS)

print("prompts.py created successfully!")

prompts.py created successfully!


In [59]:
from code_sentry.prompts import SYSTEM_PROMPT

print(SYSTEM_PROMPT[:500])


You are an AI-powered security code reviewer.

Your role is to help developers identify security vulnerabilities
in their source code.

You must:

1. Understand the user's security question.
2. Review the supplied source code.
3. Use the local scan_vulnerabilities tool when security
   vulnerabilities need to be checked.
4. Explain detected vulnerabilities in simple language.
5. Provide a practical suggested fix for every finding.
6. Never claim that code is completely secure.
7. Clearly distin


In [60]:
main_code = r'''
import os
import json

from google import genai

from code_sentry.tools import scan_vulnerabilities
from code_sentry.prompts import SYSTEM_PROMPT


MODEL_NAME = "gemini-3.8-flash"


# Gemini client
api_key = os.environ.get("GEMINI_API_KEY")

if not api_key:
    raise RuntimeError(
        "GEMINI_API_KEY is not set. "
        "Please set your Gemini API key before running the AI reviewer."
    )

client = genai.Client(api_key=api_key)


# Local tool declaration for Gemini
scan_tool = {
    "type": "function",
    "name": "scan_vulnerabilities",
    "description": (
        "Scan source code for common security vulnerabilities "
        "such as hardcoded secrets, code injection, SQL injection, "
        "command injection, insecure deserialization, weak hashing, "
        "and missing input validation."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "code": {
                "type": "string",
                "description": "The source code that should be scanned."
            },
            "language": {
                "type": "string",
                "description": "The programming language of the source code."
            }
        },
        "required": ["code"]
    }
}


def ai_security_review(user_query, code, language="unknown"):

    if not code or not code.strip():
        return "Please provide code for security review."

    user_prompt = (
        "Security Question: " + user_query +
        "\n\nProgramming Language: " + language +
        "\n\nSource Code:\n" + code +
        "\n\nReview this code for security vulnerabilities. "
        "Use the scan_vulnerabilities tool when appropriate."
    )

    try:

        interaction = client.interactions.create(
            model=MODEL_NAME,
            input=user_prompt,
            system_instruction=SYSTEM_PROMPT,
            tools=[scan_tool]
        )

        function_calls = [
            step for step in interaction.steps
            if step.type == "function_call"
        ]

        # If Gemini did not request the local scanner
        if not function_calls:
            return interaction.output_text

        function_results = []

        for call in function_calls:

            if call.name == "scan_vulnerabilities":

                arguments = call.arguments

                if isinstance(arguments, str):
                    arguments = json.loads(arguments)

                scanner_result = scan_vulnerabilities(
                    arguments.get("code", code),
                    arguments.get("language", language)
                )

                function_results.append({
                    "type": "function_result",
                    "name": call.name,
                    "call_id": call.id,
                    "result": [
                        {
                            "type": "text",
                            "text": json.dumps(scanner_result)
                        }
                    ]
                })

        final_interaction = client.interactions.create(
            model=MODEL_NAME,
            previous_interaction_id=interaction.id,
            input=function_results,
            system_instruction=SYSTEM_PROMPT,
            tools=[scan_tool]
        )

        return final_interaction.output_text

    except Exception as e:

        error_message = str(e)

        if "429" in error_message or "too_many_requests" in error_message:
            return (
                "AI REVIEW TEMPORARILY UNAVAILABLE\n\n"
                "The Gemini API request limit has been reached.\n\n"
                "The local security scanner is still available."
            )

        if "503" in error_message or "UNAVAILABLE" in error_message:
            return (
                "AI REVIEW TEMPORARILY UNAVAILABLE\n\n"
                "The Gemini service is currently busy. "
                "Please try again later."
            )

        return (
            "ERROR WHILE PERFORMING AI SECURITY REVIEW:\n"
            + error_message
        )


def run_ai_security_reviewer():

    print("=" * 60)
    print("       AI-POWERED SECURITY CODE REVIEWER")
    print("=" * 60)

    user_query = input("\nSecurity Question: ")
    language = input("Programming Language: ")

    print("\nPaste your source code.")
    print("Type END when you have finished.\n")

    code_lines = []

    while True:

        line = input()

        if line.strip() == "END":
            break

        code_lines.append(line)

    code = "\n".join(code_lines)

    if not code.strip():
        print("\nNo code provided.")
        return

    print("\nAnalyzing code...")
    print("-" * 60)

    result = ai_security_review(
        user_query,
        code,
        language
    )

    print(result)


if __name__ == "__main__":
    run_ai_security_reviewer()
'''

with open("code_sentry/main.py", "w", encoding="utf-8") as f:
    f.write(main_code)

print("main.py created successfully!")

main.py created successfully!


In [61]:
import os

for root, dirs, files in os.walk("code_sentry"):
    level = root.replace("code_sentry", "").count(os.sep)
    indent = "    " * level
    print(f"{indent}{os.path.basename(root)}/")

    for file in files:
        print(f"{indent}    {file}")

code_sentry/
    tools.py
    main.py
    prompts.py
    tests/
    __pycache__/
        tools.cpython-313.pyc
        prompts.cpython-313.pyc


In [62]:
test_code = r'''
from code_sentry.tools import scan_vulnerabilities


def test_eval_detection():
    result = scan_vulnerabilities(
        "result = eval(user_input)",
        "python"
    )
    assert any(
        f["category"] == "Code Injection"
        for f in result["findings"]
    )


def test_shell_injection():
    result = scan_vulnerabilities(
        "subprocess.run(command, shell=True)",
        "python"
    )
    assert any(
        f["category"] == "Command Injection"
        for f in result["findings"]
    )


def test_sql_injection():
    code = 'query = "SELECT * FROM users WHERE id=" + user_id'
    result = scan_vulnerabilities(code, "python")

    assert any(
        f["category"] == "SQL Injection"
        for f in result["findings"]
    )


def test_hardcoded_secret():
    code = 'password = "mypassword123"'
    result = scan_vulnerabilities(code, "python")

    assert any(
        f["category"] == "Hardcoded Secret"
        for f in result["findings"]
    )


def test_clean_code():
    code = """
def add(a, b):
    return a + b
"""

    result = scan_vulnerabilities(code, "python")

    assert result["status"] == "clean"
    assert result["findings"] == []


def test_empty_code():
    result = scan_vulnerabilities("", "python")

    assert result["status"] == "clean"
    assert result["findings"] == []


if __name__ == "__main__":
    test_eval_detection()
    test_shell_injection()
    test_sql_injection()
    test_hardcoded_secret()
    test_clean_code()
    test_empty_code()

    print("ALL TESTS PASSED!")
'''

with open("code_sentry/tests/test_tools.py", "w", encoding="utf-8") as f:
    f.write(test_code)

print("test_tools.py created successfully!")

test_tools.py created successfully!


In [63]:
import sys
sys.path.insert(0, "/content")

from code_sentry.tests.test_tools import *

test_eval_detection()
test_shell_injection()
test_sql_injection()
test_hardcoded_secret()
test_clean_code()
test_empty_code()

print("\n===================================")
print("ALL SECURITY TESTS PASSED!")
print("===================================")


ALL SECURITY TESTS PASSED!


In [64]:
import os
import shutil

# Remove automatically generated cache folders
cache_folder = "code_sentry/__pycache__"

if os.path.exists(cache_folder):
    shutil.rmtree(cache_folder)

# Remove any cache inside tests
test_cache = "code_sentry/tests/__pycache__"

if os.path.exists(test_cache):
    shutil.rmtree(test_cache)

# Create final submission ZIP
zip_path = shutil.make_archive(
    "AI_Security_Code_Reviewer",
    "zip",
    ".",
    "code_sentry"
)

print("ZIP created successfully!")
print(zip_path)

ZIP created successfully!
/content/AI_Security_Code_Reviewer.zip


In [65]:
from google.colab import files

files.download("AI_Security_Code_Reviewer.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [66]:
import os
import shutil

# Remove old ZIP if it exists
if os.path.exists("AI_Security_Code_Reviewer_Final.zip"):
    os.remove("AI_Security_Code_Reviewer_Final.zip")

# Create a clean submission folder
submission_folder = "AI_Security_Code_Reviewer"

if os.path.exists(submission_folder):
    shutil.rmtree(submission_folder)

os.makedirs(submission_folder)

# Copy project folder
shutil.copytree(
    "code_sentry",
    os.path.join(submission_folder, "code_sentry"),
    ignore=shutil.ignore_patterns("__pycache__", "*.pyc")
)

# Copy README
if os.path.exists("README.md"):
    shutil.copy(
        "README.md",
        os.path.join(submission_folder, "README.md")
    )

# Create ZIP
zip_file = shutil.make_archive(
    "AI_Security_Code_Reviewer_Final",
    "zip",
    ".",
    submission_folder
)

print("======================================")
print("FINAL SUBMISSION PACKAGE CREATED")
print("======================================")
print(zip_file)

FINAL SUBMISSION PACKAGE CREATED
/content/AI_Security_Code_Reviewer_Final.zip


In [67]:
import zipfile

with zipfile.ZipFile(
    "AI_Security_Code_Reviewer_Final.zip",
    "r"
) as z:

    print("Files inside ZIP:")
    for name in z.namelist():
        print(" -", name)

Files inside ZIP:
 - AI_Security_Code_Reviewer/
 - AI_Security_Code_Reviewer/code_sentry/
 - AI_Security_Code_Reviewer/README.md
 - AI_Security_Code_Reviewer/code_sentry/tests/
 - AI_Security_Code_Reviewer/code_sentry/tools.py
 - AI_Security_Code_Reviewer/code_sentry/main.py
 - AI_Security_Code_Reviewer/code_sentry/prompts.py
 - AI_Security_Code_Reviewer/code_sentry/tests/test_tools.py


In [68]:
import os

required_files = [
    "README.md",
    "code_sentry/main.py",
    "code_sentry/tools.py",
    "code_sentry/prompts.py",
    "code_sentry/tests/test_tools.py"
]

print("PROJECT VERIFICATION")
print("=" * 50)

all_ok = True

for file in required_files:
    if os.path.exists(file):
        print("✓", file)
    else:
        print("✗ MISSING:", file)
        all_ok = False

print("=" * 50)

if all_ok:
    print("PROJECT STRUCTURE: READY")
else:
    print("PROJECT STRUCTURE: INCOMPLETE")

PROJECT VERIFICATION
✓ README.md
✓ code_sentry/main.py
✓ code_sentry/tools.py
✓ code_sentry/prompts.py
✓ code_sentry/tests/test_tools.py
PROJECT STRUCTURE: READY


In [69]:
import sys
sys.path.insert(0, "/content")

from code_sentry.tests.test_tools import *

test_eval_detection()
test_shell_injection()
test_sql_injection()
test_hardcoded_secret()
test_clean_code()
test_empty_code()

print("\n✓ ALL TESTS PASSED")


✓ ALL TESTS PASSED


In [70]:
from code_sentry.tools import scan_vulnerabilities

sample_code = '''
password = "admin123"

user_input = input("Enter value: ")

result = eval(user_input)
'''

result = scan_vulnerabilities(sample_code, "python")

print("=" * 60)
print("SECURITY SCAN RESULT")
print("=" * 60)

print("Status:", result["status"])
print("Language:", result["language"])
print("Number of findings:", len(result["findings"]))

for finding in result["findings"]:
    print("\nCategory:", finding["category"])
    print("Severity:", finding["severity"])
    print("Line:", finding["line"])
    print("Problem:", finding["description"])
    print("Fix:", finding["suggested_fix"])

SECURITY SCAN RESULT
Status: vulnerabilities_found
Language: python
Number of findings: 3

Category: Hardcoded Secret
Severity: High
Line: 2
Problem: A password, API key, token, or secret appears to be hardcoded.
Fix: Store secrets in environment variables or a secure secret manager.

Category: Code Injection
Severity: High
Line: 6
Problem: eval() can execute dynamically supplied code.
Fix: Avoid eval() and use safe parsing or validated input.

Category: Missing Input Validation
Severity: Medium
Line: 4
Problem: User-controlled input appears to be used without obvious validation.
Fix: Validate the type, length, format, and allowed values.


In [71]:
demo_code = r'''
from code_sentry.tools import scan_vulnerabilities


print("=" * 60)
print("       AI-POWERED SECURITY CODE REVIEWER")
print("=" * 60)

code = """
password = "admin123"

user_input = input("Enter expression: ")

result = eval(user_input)
"""

print("\nSource Code:")
print(code)

print("-" * 60)
print("Running security scanner...")
print("-" * 60)

result = scan_vulnerabilities(code, "python")

findings = result["findings"]

if not findings:
    verdict = "Safe"
elif any(f["severity"] == "High" for f in findings):
    verdict = "High Risk"
else:
    verdict = "Needs Attention"

print("\nSummary Verdict:")
print(verdict)

print("\nFindings:")

for i, finding in enumerate(findings, start=1):
    print(f"\nFinding {i}")
    print("Category:", finding["category"])
    print("Severity:", finding["severity"])
    print("Line:", finding["line"])
    print("Problem:", finding["description"])
    print("Suggested Fix:", finding["suggested_fix"])

print("\nExplanation:")
print(
    "The scanner detected security patterns that may expose "
    "the application to attacks. The suggested fixes should "
    "be applied before deploying the code."
)

print("\n" + "=" * 60)
print("DEMO COMPLETED")
print("=" * 60)
'''

with open("code_sentry/demo.py", "w", encoding="utf-8") as f:
    f.write(demo_code)

print("demo.py created successfully!")

demo.py created successfully!


In [72]:
%run code_sentry/demo.py

       AI-POWERED SECURITY CODE REVIEWER

Source Code:

password = "admin123"

user_input = input("Enter expression: ")

result = eval(user_input)

------------------------------------------------------------
Running security scanner...
------------------------------------------------------------

Summary Verdict:
High Risk

Findings:

Finding 1
Category: Hardcoded Secret
Severity: High
Line: 2
Problem: A password, API key, token, or secret appears to be hardcoded.
Suggested Fix: Store secrets in environment variables or a secure secret manager.

Finding 2
Category: Code Injection
Severity: High
Line: 6
Problem: eval() can execute dynamically supplied code.
Suggested Fix: Avoid eval() and use safe parsing or validated input.

Finding 3
Category: Missing Input Validation
Severity: Medium
Line: 4
Problem: User-controlled input appears to be used without obvious validation.
Suggested Fix: Validate the type, length, format, and allowed values.

Explanation:
The scanner detected security pa